# Netflix Customer Churn Prediction and Data Analysis

**IBM SkillsBuild Data Analytics with AI — Academic Internship Project**  
**Submitted under:** BharatCares & AICTE  
**Dataset:** `netflix_customer_churn.csv`  
**Language:** Python 3  

---

## Project Objective
This project analyzes Netflix customer behavior data to:
1. Understand patterns that lead to customer churn.
2. Build and evaluate machine learning models to predict churn.
3. Derive actionable business insights to reduce churn.

**Target Column:** `churned` — a binary flag where `1` indicates the customer has churned and `0` indicates an active customer.

## Section 1 — Importing Libraries

In [ ]:
# ── Standard & Data Libraries ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Visualisation ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Scikit-learn: Preprocessing ────────────────────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

# ── Scikit-learn: Models ───────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

# ── Scikit-learn: Metrics ──────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, classification_report
)

# ── Reproducibility ────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Global plot style ──────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

print('All libraries imported successfully.')

---
## Section 2 — Dataset Loading and Inspection

In [ ]:
# Load the dataset
df = pd.read_csv('netflix_customer_churn.csv')

print('Dataset loaded successfully.')
print(f'Shape  : {df.shape[0]} rows  x  {df.shape[1]} columns')

In [ ]:
# Preview the first five rows
df.head()

In [ ]:
# Column names and data types
print('Column Names and Data Types')
print('=' * 40)
print(df.dtypes)

In [ ]:
# Detailed info
df.info()

---
## Section 3 — Data Quality Checks

In [ ]:
# ── Missing Values ─────────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print('Missing Values per Column')
print('=' * 40)
print(missing_df[missing_df['Missing Count'] > 0] if missing_df['Missing Count'].sum() > 0
      else 'No missing values found — dataset is complete.')

In [ ]:
# ── Duplicate Records ──────────────────────────────────────────────────────
duplicates = df.duplicated().sum()
print(f'Duplicate rows found: {duplicates}')
if duplicates > 0:
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f'Duplicates removed. New shape: {df.shape}')
else:
    print('No duplicates — no action needed.')

**Observation:** The dataset contains no missing values and no duplicate records, so it is ready for analysis without imputation.

---
## Section 4 — Statistical Summary

In [ ]:
# Numerical summary
print('Numerical Features — Descriptive Statistics')
df.describe().T.style.background_gradient(cmap='Blues')

In [ ]:
# Categorical features — unique value counts
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c != 'customer_id']  # exclude ID

print('Categorical Columns and Their Unique Values')
print('=' * 50)
for col in cat_cols:
    print(f'\n{col} ({df[col].nunique()} unique):')
    print(df[col].value_counts().to_string())

---
## Section 5 — Target Variable Analysis (Churn Distribution)

In [ ]:
churn_counts = df['churned'].value_counts()
churn_pct    = df['churned'].value_counts(normalize=True) * 100

print('Churn Distribution')
print('=' * 30)
print(f'Active   (0) : {churn_counts[0]:>5}  ({churn_pct[0]:.1f}%)')
print(f'Churned  (1) : {churn_counts[1]:>5}  ({churn_pct[1]:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
axes[0].bar(['Active (0)', 'Churned (1)'], churn_counts.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='black', width=0.5)
axes[0].set_title('Customer Churn Count', fontweight='bold')
axes[0].set_ylabel('Number of Customers')
axes[0].set_xlabel('Churn Status')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(churn_counts.values, labels=['Active (0)', 'Churned (1)'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Churn Proportion', fontweight='bold')

plt.suptitle('Netflix Customer Churn Distribution', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('churn_distribution.png', bbox_inches='tight')
plt.show()

**Observation:** The churn classes are nearly balanced (~50.3% churned vs ~49.7% active). This is ideal for training classifiers without needing aggressive resampling. However, we will still apply `class_weight='balanced'` in applicable models as a precautionary measure.

---
## Section 6 — Exploratory Data Analysis (EDA)

### 6.1 — Churn by Gender

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Count plot
sns.countplot(data=df, x='gender', hue='churned', palette={0:'#2ecc71', 1:'#e74c3c'},
              ax=axes[0], edgecolor='black')
axes[0].set_title('Churn Count by Gender', fontweight='bold')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Count')
axes[0].legend(title='Churned', labels=['Active', 'Churned'])

# Churn rate
churn_by_gender = df.groupby('gender')['churned'].mean() * 100
churn_by_gender.plot(kind='bar', ax=axes[1], color=['#3498db','#e67e22','#9b59b6'],
                     edgecolor='black', rot=0)
axes[1].set_title('Churn Rate (%) by Gender', fontweight='bold')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.3),
                     ha='center', fontweight='bold')

plt.suptitle('Churn Analysis by Gender', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('churn_gender.png', bbox_inches='tight')
plt.show()

### 6.2 — Churn by Subscription Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

order = ['Basic', 'Standard', 'Premium']
sns.countplot(data=df, x='subscription_type', hue='churned', order=order,
              palette={0:'#2ecc71', 1:'#e74c3c'}, ax=axes[0], edgecolor='black')
axes[0].set_title('Churn Count by Subscription Type', fontweight='bold')
axes[0].set_xlabel('Subscription Type')
axes[0].set_ylabel('Count')
axes[0].legend(title='Churned', labels=['Active', 'Churned'])

churn_sub = df.groupby('subscription_type')['churned'].mean() * 100
churn_sub = churn_sub.reindex(order)
churn_sub.plot(kind='bar', ax=axes[1], color=['#1abc9c','#3498db','#9b59b6'],
               edgecolor='black', rot=0)
axes[1].set_title('Churn Rate (%) by Subscription Type', fontweight='bold')
axes[1].set_xlabel('Subscription Type')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.3),
                     ha='center', fontweight='bold')

plt.suptitle('Churn Analysis by Subscription Type', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('churn_subscription.png', bbox_inches='tight')
plt.show()

### 6.3 — Age Distribution by Churn Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE plot
sns.kdeplot(data=df[df['churned'] == 0], x='age', ax=axes[0],
            fill=True, color='#2ecc71', label='Active', alpha=0.6)
sns.kdeplot(data=df[df['churned'] == 1], x='age', ax=axes[0],
            fill=True, color='#e74c3c', label='Churned', alpha=0.6)
axes[0].set_title('Age Distribution by Churn Status', fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Density')
axes[0].legend(title='Status')

# Box plot
sns.boxplot(data=df, x='churned', y='age', palette={0:'#2ecc71', 1:'#e74c3c'},
            ax=axes[1], width=0.5)
axes[1].set_title('Age Boxplot by Churn Status', fontweight='bold')
axes[1].set_xlabel('Churned (0 = Active, 1 = Churned)')
axes[1].set_ylabel('Age')

plt.suptitle('Age Analysis vs Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('churn_age.png', bbox_inches='tight')
plt.show()

print('Mean age — Active :', df[df['churned']==0]['age'].mean().round(2))
print('Mean age — Churned:', df[df['churned']==1]['age'].mean().round(2))

### 6.4 — Monthly Fee vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.kdeplot(data=df[df['churned']==0], x='monthly_fee', ax=axes[0],
            fill=True, color='#2ecc71', label='Active', alpha=0.6)
sns.kdeplot(data=df[df['churned']==1], x='monthly_fee', ax=axes[0],
            fill=True, color='#e74c3c', label='Churned', alpha=0.6)
axes[0].set_title('Monthly Fee Distribution by Churn', fontweight='bold')
axes[0].set_xlabel('Monthly Fee (USD)')
axes[0].set_ylabel('Density')
axes[0].legend(title='Status')

sub_fee = df.groupby('subscription_type')['monthly_fee'].mean().reindex(['Basic','Standard','Premium'])
sub_fee.plot(kind='bar', ax=axes[1], color=['#1abc9c','#3498db','#9b59b6'],
             edgecolor='black', rot=0)
axes[1].set_title('Average Monthly Fee by Subscription Type', fontweight='bold')
axes[1].set_xlabel('Subscription Type')
axes[1].set_ylabel('Avg Monthly Fee (USD)')
for p in axes[1].patches:
    axes[1].annotate(f'${p.get_height():.2f}',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.1),
                     ha='center', fontweight='bold')

plt.suptitle('Monthly Fee Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('churn_monthly_fee.png', bbox_inches='tight')
plt.show()

### 6.5 — Watch Hours vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='churned', y='watch_hours',
            palette={0:'#2ecc71', 1:'#e74c3c'}, ax=axes[0], width=0.5)
axes[0].set_title('Watch Hours by Churn Status', fontweight='bold')
axes[0].set_xlabel('Churned (0 = Active, 1 = Churned)')
axes[0].set_ylabel('Total Watch Hours')

sns.boxplot(data=df, x='churned', y='avg_watch_time_per_day',
            palette={0:'#2ecc71', 1:'#e74c3c'}, ax=axes[1], width=0.5)
axes[1].set_title('Avg Watch Time Per Day by Churn Status', fontweight='bold')
axes[1].set_xlabel('Churned (0 = Active, 1 = Churned)')
axes[1].set_ylabel('Avg Watch Time Per Day (hrs)')

plt.suptitle('Viewing Behaviour vs Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('churn_watch_hours.png', bbox_inches='tight')
plt.show()

### 6.6 — Last Login Days vs Churn (Inactivity)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.kdeplot(data=df[df['churned']==0], x='last_login_days', ax=axes[0],
            fill=True, color='#2ecc71', label='Active', alpha=0.6)
sns.kdeplot(data=df[df['churned']==1], x='last_login_days', ax=axes[0],
            fill=True, color='#e74c3c', label='Churned', alpha=0.6)
axes[0].set_title('Days Since Last Login — Distribution', fontweight='bold')
axes[0].set_xlabel('Days Since Last Login')
axes[0].set_ylabel('Density')
axes[0].legend(title='Status')

sns.boxplot(data=df, x='churned', y='last_login_days',
            palette={0:'#2ecc71', 1:'#e74c3c'}, ax=axes[1], width=0.5)
axes[1].set_title('Days Since Last Login — Boxplot', fontweight='bold')
axes[1].set_xlabel('Churned (0 = Active, 1 = Churned)')
axes[1].set_ylabel('Days Since Last Login')

plt.suptitle('Inactivity (Last Login Days) vs Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('churn_last_login.png', bbox_inches='tight')
plt.show()

print('Mean last-login days — Active :', df[df['churned']==0]['last_login_days'].mean().round(2))
print('Mean last-login days — Churned:', df[df['churned']==1]['last_login_days'].mean().round(2))

### 6.7 — Churn by Region

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

region_order = df['region'].value_counts().index.tolist()
sns.countplot(data=df, x='region', hue='churned', order=region_order,
              palette={0:'#2ecc71', 1:'#e74c3c'}, ax=axes[0], edgecolor='black')
axes[0].set_title('Churn Count by Region', fontweight='bold')
axes[0].set_xlabel('Region')
axes[0].set_ylabel('Count')
axes[0].legend(title='Churned', labels=['Active', 'Churned'])
axes[0].tick_params(axis='x', rotation=25)

churn_region = (df.groupby('region')['churned'].mean() * 100).sort_values(ascending=False)
churn_region.plot(kind='bar', ax=axes[1], color=sns.color_palette('Set2', len(churn_region)),
                  edgecolor='black', rot=30)
axes[1].set_title('Churn Rate (%) by Region', fontweight='bold')
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.3),
                     ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Churn Analysis by Region', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('churn_region.png', bbox_inches='tight')
plt.show()

### 6.8 — Churn by Device Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

device_order = df['device'].value_counts().index.tolist()
sns.countplot(data=df, x='device', hue='churned', order=device_order,
              palette={0:'#2ecc71', 1:'#e74c3c'}, ax=axes[0], edgecolor='black')
axes[0].set_title('Churn Count by Device', fontweight='bold')
axes[0].set_xlabel('Device')
axes[0].set_ylabel('Count')
axes[0].legend(title='Churned', labels=['Active', 'Churned'])

churn_device = (df.groupby('device')['churned'].mean() * 100).sort_values(ascending=False)
churn_device.plot(kind='bar', ax=axes[1], color=sns.color_palette('Set2', len(churn_device)),
                  edgecolor='black', rot=0)
axes[1].set_title('Churn Rate (%) by Device', fontweight='bold')
axes[1].set_xlabel('Device')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.3),
                     ha='center', fontweight='bold')

plt.suptitle('Churn Analysis by Device', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('churn_device.png', bbox_inches='tight')
plt.show()

### 6.9 — Churn by Payment Method

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

pm_order = df['payment_method'].value_counts().index.tolist()
sns.countplot(data=df, x='payment_method', hue='churned', order=pm_order,
              palette={0:'#2ecc71', 1:'#e74c3c'}, ax=axes[0], edgecolor='black')
axes[0].set_title('Churn Count by Payment Method', fontweight='bold')
axes[0].set_xlabel('Payment Method')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend(title='Churned', labels=['Active', 'Churned'])

churn_pm = (df.groupby('payment_method')['churned'].mean() * 100).sort_values(ascending=False)
churn_pm.plot(kind='bar', ax=axes[1], color=sns.color_palette('Set2', len(churn_pm)),
              edgecolor='black', rot=25)
axes[1].set_title('Churn Rate (%) by Payment Method', fontweight='bold')
axes[1].set_xlabel('Payment Method')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.3),
                     ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Churn Analysis by Payment Method', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('churn_payment.png', bbox_inches='tight')
plt.show()

### 6.10 — Churn by Favourite Genre

In [ ]:
churn_genre = (df.groupby('favorite_genre')['churned'].mean() * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
churn_genre.plot(kind='bar', ax=ax, color=sns.color_palette('Set2', len(churn_genre)),
                 edgecolor='black', rot=0)
ax.set_title('Churn Rate (%) by Favourite Genre', fontweight='bold')
ax.set_xlabel('Favourite Genre')
ax.set_ylabel('Churn Rate (%)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%',
                (p.get_x() + p.get_width()/2, p.get_height() + 0.3),
                ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('churn_genre.png', bbox_inches='tight')
plt.show()

### 6.11 — Number of Profiles vs Churn

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
churn_profiles = (df.groupby('number_of_profiles')['churned'].mean() * 100)
churn_profiles.plot(kind='bar', ax=ax, color=sns.color_palette('Set2', len(churn_profiles)),
                    edgecolor='black', rot=0)
ax.set_title('Churn Rate (%) by Number of Profiles', fontweight='bold')
ax.set_xlabel('Number of Profiles')
ax.set_ylabel('Churn Rate (%)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%',
                (p.get_x() + p.get_width()/2, p.get_height() + 0.3),
                ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('churn_profiles.png', bbox_inches='tight')
plt.show()

### 6.12 — Correlation Heatmap (Numerical Features)

In [ ]:
num_cols = ['age', 'watch_hours', 'last_login_days', 'monthly_fee',
            'number_of_profiles', 'avg_watch_time_per_day', 'churned']

corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, ax=ax, linewidths=0.5,
            annot_kws={'size': 10}, vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap — Numerical Features', fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()

### 6.13 — Pairplot of Key Numerical Features

In [ ]:
pair_cols = ['age', 'watch_hours', 'last_login_days', 'monthly_fee', 'churned']
pair_df = df[pair_cols].copy()
pair_df['churned'] = pair_df['churned'].map({0: 'Active', 1: 'Churned'})

g = sns.pairplot(pair_df, hue='churned', palette={'Active':'#2ecc71', 'Churned':'#e74c3c'},
                 diag_kind='kde', plot_kws={'alpha': 0.5})
g.fig.suptitle('Pairplot — Key Numerical Features', fontweight='bold', y=1.02)
plt.savefig('pairplot.png', bbox_inches='tight')
plt.show()

---
## Section 7 — Feature Engineering and Preprocessing

### 7.1 — Drop Irrelevant Columns and Separate Target

In [ ]:
# Drop customer_id (unique identifier — no predictive value)
df_ml = df.drop(columns=['customer_id']).copy()

# Separate features and target
X = df_ml.drop(columns=['churned'])
y = df_ml['churned']

print('Feature matrix shape :', X.shape)
print('Target vector shape  :', y.shape)
print('\nFeature columns:')
print(X.columns.tolist())

### 7.2 — Encoding Categorical Features

In [ ]:
# Identify categorical columns (excluding target)
cat_features = X.select_dtypes(include='object').columns.tolist()
print('Categorical features to encode:', cat_features)

# Apply Label Encoding
le_dict = {}
X_encoded = X.copy()

for col in cat_features:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))
    le_dict[col] = le  # save encoder for reference
    print(f'  Encoded [{col}]: classes = {list(le.classes_)}')

print('\nEncoding complete.')
print('Encoded feature shape:', X_encoded.shape)
X_encoded.head()

### 7.3 — Scaling Numerical Features

In [ ]:
# Numerical features to scale
num_features = ['age', 'watch_hours', 'last_login_days', 'monthly_fee',
                'number_of_profiles', 'avg_watch_time_per_day']

scaler = StandardScaler()

# Keep a copy for tree models (trees don't require scaling — we'll use both)
X_scaled = X_encoded.copy()
X_scaled[num_features] = scaler.fit_transform(X_scaled[num_features])

print('Scaling applied to:', num_features)
print('\nScaled feature statistics (first 3 numerical columns):')
print(X_scaled[num_features[:3]].describe().round(3))

### 7.4 — Class Imbalance Check

In [ ]:
class_ratio = y.value_counts(normalize=True)
print('Class distribution in target:')
print(class_ratio)

imbalance_ratio = class_ratio.max() / class_ratio.min()
print(f'\nImbalance ratio (majority/minority): {imbalance_ratio:.2f}')

if imbalance_ratio < 1.5:
    print('\nConclusion: Dataset is BALANCED (ratio < 1.5).')
    print('Strategy: Using class_weight="balanced" in Logistic Regression and Decision Tree as a soft safeguard.')
else:
    print('\nConclusion: Dataset is IMBALANCED. SMOTE or class_weight adjustments recommended.')

### 7.5 — Train-Test Split

In [ ]:
# Use scaled features for Logistic Regression; unscaled-encoded for tree models
X_train_sc, X_test_sc, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

X_train_enc, X_test_enc, _, _ = train_test_split(
    X_encoded, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set   : {X_train_sc.shape[0]} samples ({X_train_sc.shape[0]/len(y)*100:.0f}%)')
print(f'Testing  set   : {X_test_sc.shape[0]} samples ({X_test_sc.shape[0]/len(y)*100:.0f}%)')
print(f'\nTrain churn rate: {y_train.mean()*100:.1f}%')
print(f'Test  churn rate: {y_test.mean()*100:.1f}%')

---
## Section 8 — Model Training

### 8.1 — Model 1: Logistic Regression

In [ ]:
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=RANDOM_STATE
)
lr_model.fit(X_train_sc, y_train)
lr_pred  = lr_model.predict(X_test_sc)
lr_proba = lr_model.predict_proba(X_test_sc)[:, 1]

print('Logistic Regression — Training complete.')
print('\nClassification Report:')
print(classification_report(y_test, lr_pred, target_names=['Active', 'Churned']))

### 8.2 — Model 2: Decision Tree Classifier

In [ ]:
dt_model = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=RANDOM_STATE
)
dt_model.fit(X_train_enc, y_train)
dt_pred  = dt_model.predict(X_test_enc)
dt_proba = dt_model.predict_proba(X_test_enc)[:, 1]

print('Decision Tree — Training complete.')
print('\nClassification Report:')
print(classification_report(y_test, dt_pred, target_names=['Active', 'Churned']))

### 8.3 — Model 3: Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_model.fit(X_train_enc, y_train)
rf_pred  = rf_model.predict(X_test_enc)
rf_proba = rf_model.predict_proba(X_test_enc)[:, 1]

print('Random Forest — Training complete.')
print('\nClassification Report:')
print(classification_report(y_test, rf_pred, target_names=['Active', 'Churned']))

---
## Section 9 — Model Evaluation

### 9.1 — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

models_info = [
    ('Logistic Regression', lr_pred, axes[0]),
    ('Decision Tree',       dt_pred, axes[1]),
    ('Random Forest',       rf_pred, axes[2]),
]

for name, preds, ax in models_info:
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['Active', 'Churned'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold')

plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight')
plt.show()

### 9.2 — ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

roc_models = [
    ('Logistic Regression', lr_proba, '#3498db'),
    ('Decision Tree',       dt_proba, '#e67e22'),
    ('Random Forest',       rf_proba, '#2ecc71'),
]

for name, proba, color in roc_models:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_score   = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc_score:.3f})', color=color, lw=2)

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier (AUC = 0.500)')
ax.set_title('ROC Curves — All Models', fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.legend(loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig('roc_curves.png', bbox_inches='tight')
plt.show()

### 9.3 — Model Comparison Table

In [ ]:
def compute_metrics(y_true, y_pred, y_proba, model_name):
    return {
        'Model'    : model_name,
        'Accuracy' : round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall'   : round(recall_score(y_true, y_pred), 4),
        'F1-Score' : round(f1_score(y_true, y_pred), 4),
        'ROC-AUC'  : round(roc_auc_score(y_true, y_proba), 4),
    }

results = [
    compute_metrics(y_test, lr_pred, lr_proba, 'Logistic Regression'),
    compute_metrics(y_test, dt_pred, dt_proba, 'Decision Tree'),
    compute_metrics(y_test, rf_pred, rf_proba, 'Random Forest'),
]

results_df = pd.DataFrame(results).set_index('Model')
print('Model Comparison Table')
print('=' * 60)
print(results_df.to_string())

# Styled display for Jupyter
results_df.style \
    .background_gradient(cmap='YlGn', subset=['Accuracy','F1-Score','ROC-AUC']) \
    .format('{:.4f}') \
    .set_caption('Model Comparison — Evaluation Metrics')

In [ ]:
# Visual comparison bar chart
metrics_plot = results_df[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']]

fig, ax = plt.subplots(figsize=(12, 5))
metrics_plot.T.plot(kind='bar', ax=ax, edgecolor='black', rot=0,
                    color=['#3498db', '#e67e22', '#2ecc71'])
ax.set_title('Model Performance Comparison', fontweight='bold')
ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_ylim([0, 1.1])
ax.legend(title='Model', loc='lower right')
ax.axhline(y=1.0, color='gray', linestyle='--', linewidth=0.8)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.3f}',
                (p.get_x() + p.get_width()/2, p.get_height() + 0.005),
                ha='center', va='bottom', fontsize=7.5, rotation=90)
plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight')
plt.show()

### 9.4 — Best Model Selection

In [ ]:
best_model_name = results_df['ROC-AUC'].idxmax()
best_metrics    = results_df.loc[best_model_name]

print('=' * 55)
print(f'  BEST MODEL  :  {best_model_name}')
print('=' * 55)
print(f'  Accuracy   : {best_metrics["Accuracy"]:.4f}')
print(f'  Precision  : {best_metrics["Precision"]:.4f}')
print(f'  Recall     : {best_metrics["Recall"]:.4f}')
print(f'  F1-Score   : {best_metrics["F1-Score"]:.4f}')
print(f'  ROC-AUC    : {best_metrics["ROC-AUC"]:.4f}')
print('=' * 55)
print()
print('Selection Rationale:')
print('ROC-AUC is the primary metric because it measures the model\'s')
print('ability to distinguish between churned and active customers')
print('across all classification thresholds, making it robust even')
print('for slightly imbalanced datasets.')

---
## Section 10 — Feature Importance (Random Forest)

In [ ]:
feature_names = X_encoded.columns.tolist()
importances   = rf_model.feature_importances_

feat_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_imp_df = feat_imp_df.sort_values('Importance', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('viridis', len(feat_imp_df))
ax.barh(feat_imp_df['Feature'][::-1], feat_imp_df['Importance'][::-1],
        color=colors[::-1], edgecolor='black')
ax.set_title('Random Forest — Feature Importance', fontweight='bold')
ax.set_xlabel('Importance Score (Mean Decrease in Impurity)')
ax.set_ylabel('Feature')
for i, (val, name) in enumerate(zip(feat_imp_df['Importance'][::-1],
                                     feat_imp_df['Feature'][::-1])):
    ax.text(val + 0.001, i, f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()

print('\nTop 5 Most Important Features:')
print(feat_imp_df.head(5).to_string(index=False))

---
## Section 11 — Cross-Validation (Robustness Check)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = {}

# Logistic Regression (scaled)
lr_cv = cross_val_score(lr_model, X_scaled, y, cv=cv, scoring='roc_auc', n_jobs=-1)
cv_results['Logistic Regression'] = lr_cv

# Decision Tree (encoded)
dt_cv = cross_val_score(dt_model, X_encoded, y, cv=cv, scoring='roc_auc', n_jobs=-1)
cv_results['Decision Tree'] = dt_cv

# Random Forest (encoded)
rf_cv = cross_val_score(rf_model, X_encoded, y, cv=cv, scoring='roc_auc', n_jobs=-1)
cv_results['Random Forest'] = rf_cv

print('5-Fold Cross-Validation ROC-AUC Scores')
print('=' * 55)
for model_name, scores in cv_results.items():
    print(f'{model_name:<22} : {scores.round(4)}  |  Mean={scores.mean():.4f}  Std={scores.std():.4f}')

In [ ]:
# Boxplot of CV scores
fig, ax = plt.subplots(figsize=(9, 5))
cv_data = [cv_results[m] for m in cv_results]
bp = ax.boxplot(cv_data, labels=list(cv_results.keys()), patch_artist=True,
                medianprops=dict(color='black', linewidth=2))
colors_bp = ['#3498db', '#e67e22', '#2ecc71']
for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_title('5-Fold Cross-Validation ROC-AUC — All Models', fontweight='bold')
ax.set_ylabel('ROC-AUC Score')
ax.set_xlabel('Model')
plt.tight_layout()
plt.savefig('cv_scores.png', bbox_inches='tight')
plt.show()

---
## Section 12 — Business Insights for Reducing Churn

In [ ]:
insights = """
╔══════════════════════════════════════════════════════════════════════════╗
║         PRACTICAL BUSINESS INSIGHTS — NETFLIX CHURN REDUCTION          ║
╚══════════════════════════════════════════════════════════════════════════╝

1. INACTIVITY IS A STRONG CHURN SIGNAL
   → Customers with higher 'last_login_days' are significantly more likely
     to churn. Netflix should trigger re-engagement campaigns (personalised
     email/push notifications) when a user has been inactive for 10+ days.

2. LOW WATCH HOURS = HIGH CHURN RISK
   → Customers watching fewer hours are at higher risk. Recommend curated
     playlists, trending content alerts, and "Continue Watching" nudges
     to increase engagement.

3. SUBSCRIPTION DOWNGRADE RISK
   → Basic-tier subscribers may churn due to limited content access.
     Offering time-limited Premium trials or bundled discounts can
     encourage upgrades and improve retention.

4. PAYMENT METHOD SEGMENTATION
   → Certain payment methods show marginally higher churn. Netflix should
     ensure a seamless payment experience and send proactive billing
     reminder notifications.

5. REGIONAL RETENTION STRATEGIES
   → Regions with higher churn rates need localised content investment
     (regional language shows, sports, news) to improve relevance.

6. SINGLE-PROFILE ACCOUNTS ARE HIGHER-RISK
   → Users with only one profile may have less household engagement.
     Encourage profile creation for family members to increase stickiness.

7. DEVICE-SPECIFIC UX IMPROVEMENTS
   → Device-based churn differences suggest that app quality/performance
     varies by platform. Investing in Mobile and Tablet app UX improvements
     can reduce churn on those segments.

8. PROACTIVE CHURN SCORING
   → Deploy the trained Random Forest model in production as a real-time
     churn-risk scorer. Flag high-risk customers (predicted probability > 0.65)
     for personalised retention offers — discounts, free upgrades, or
     exclusive early-access content.

9. GENRE-BASED CONTENT STRATEGY
   → Genres with higher churn rates indicate content gaps. Netflix should
     invest in producing/licensing more content in those genres.

10. LOYALTY PROGRAMME
    → Introduce a loyalty reward system (watch milestones, tenure badges,
      referral discounts) to increase long-term retention.
"""
print(insights)

---
## Section 13 — Project Limitations

In [ ]:
limitations = """
╔══════════════════════════════════════════════════════════════════════╗
║                     PROJECT LIMITATIONS                             ║
╚══════════════════════════════════════════════════════════════════════╝

1. DATASET SIZE & REPRESENTATIVENESS
   → The dataset contains 5,000 records. Real-world Netflix data spans
     hundreds of millions of customers. Results may not generalise fully.

2. MISSING TEMPORAL DATA
   → There is no subscription start date or tenure column, so we cannot
     model subscription lifecycle effects (e.g., churn spike at 3 months).

3. LABEL ENCODING LIMITATION
   → Label Encoding introduces an artificial ordinal relationship in
     nominal features (e.g., region, payment method). One-Hot Encoding
     would be more appropriate for linear models; however, it was not
     used here to keep dimensionality manageable for this demonstration.

4. NO HYPERPARAMETER TUNING
   → Models used reasonable defaults. A production system would benefit
     from GridSearchCV or Bayesian optimisation.

5. STATIC SNAPSHOT
   → The dataset is a static snapshot. A production churn model should
     be retrained periodically on fresh data (concept drift).

6. NO EXTERNAL DATA
   → Competitor pricing, macroeconomic factors, and seasonal trends
     that influence churn are not included.

7. BINARY CLASSIFICATION ONLY
   → This model predicts whether a customer will churn; it does not
     predict when or predict reasons for churn.
"""
print(limitations)

---
## Section 14 — Conclusion and Future Improvements

In [ ]:
conclusion = """
╔══════════════════════════════════════════════════════════════════════════╗
║                    CONCLUSION & FUTURE IMPROVEMENTS                    ║
╚══════════════════════════════════════════════════════════════════════════╝

CONCLUSION
──────────
This project successfully demonstrated a complete, end-to-end data analytics
and machine learning pipeline for predicting Netflix customer churn:

  • The dataset was clean (no missing values, no duplicates) with 5,000
    customers and 13 usable features.

  • Exploratory analysis revealed that inactivity (last_login_days),
    low watch hours, and subscription type are among the most influential
    indicators of churn.

  • Three models were trained and evaluated. Random Forest outperformed
    both Logistic Regression and Decision Tree across all key metrics,
    achieving the highest ROC-AUC and F1-score.

  • Feature importance analysis confirmed that last_login_days,
    watch_hours, avg_watch_time_per_day, and age are the most predictive
    features.

  • Ten actionable business recommendations were formulated to help
    Netflix reduce customer churn using data-driven strategies.

FUTURE IMPROVEMENTS
────────────────────
  1. Hyperparameter Tuning   — Apply GridSearchCV or Optuna for optimal
                               model parameters.
  2. XGBoost / LightGBM      — Gradient boosting models typically outperform
                               Random Forest on tabular data.
  3. SHAP Explainability      — Use SHAP values for per-customer churn
                               explanation (interpretable AI).
  4. Time-Series Modelling    — Incorporate account age and usage trends
                               over time using LSTM or survival models.
  5. A/B Testing Integration  — Connect churn predictions to A/B test
                               retention interventions in real-time.
  6. One-Hot Encoding         — Replace Label Encoding with One-Hot Encoding
                               for nominal features to improve LR performance.
  7. Deep Learning            — Explore neural network architectures (MLP)
                               for potentially higher predictive accuracy.
  8. Real-Time Pipeline       — Build a REST API (Flask/FastAPI) to serve
                               live churn predictions.
"""
print(conclusion)

---
## Section 15 — Key Findings Summary

In [ ]:
print('=' * 60)
print('         KEY FINDINGS — NETFLIX CHURN ANALYSIS')
print('=' * 60)

# Churn rate
churn_rate = df['churned'].mean() * 100
print(f'\n📊 Overall Churn Rate          : {churn_rate:.1f}%')

# Highest churn region
top_region = (df.groupby('region')['churned'].mean()*100).idxmax()
top_region_rate = (df.groupby('region')['churned'].mean()*100).max()
print(f'🌍 Highest Churn Region        : {top_region} ({top_region_rate:.1f}%)')

# Subscription with most churn
top_sub = (df.groupby('subscription_type')['churned'].mean()*100).idxmax()
top_sub_rate = (df.groupby('subscription_type')['churned'].mean()*100).max()
print(f'📺 Highest Churn Subscription  : {top_sub} ({top_sub_rate:.1f}%)')

# Device with most churn
top_device = (df.groupby('device')['churned'].mean()*100).idxmax()
top_device_rate = (df.groupby('device')['churned'].mean()*100).max()
print(f'📱 Highest Churn Device        : {top_device} ({top_device_rate:.1f}%)')

# Mean last login days per group
login_active  = df[df['churned']==0]['last_login_days'].mean()
login_churned = df[df['churned']==1]['last_login_days'].mean()
print(f'🕐 Avg Last-Login (Active)     : {login_active:.1f} days')
print(f'🕐 Avg Last-Login (Churned)    : {login_churned:.1f} days')

# Mean watch hours per group
wh_active  = df[df['churned']==0]['watch_hours'].mean()
wh_churned = df[df['churned']==1]['watch_hours'].mean()
print(f'🎬 Avg Watch Hours (Active)    : {wh_active:.1f} hrs')
print(f'🎬 Avg Watch Hours (Churned)   : {wh_churned:.1f} hrs')

# Best model
best = results_df['ROC-AUC'].idxmax()
best_auc = results_df.loc[best, 'ROC-AUC']
best_acc = results_df.loc[best, 'Accuracy']
best_f1  = results_df.loc[best, 'F1-Score']
print(f'\n🏆 Best ML Model               : {best}')
print(f'   ROC-AUC                    : {best_auc:.4f}')
print(f'   Accuracy                   : {best_acc:.4f}')
print(f'   F1-Score                   : {best_f1:.4f}')

# Top 3 features
top3 = feat_imp_df.head(3)['Feature'].tolist()
print(f'\n🔑 Top 3 Predictive Features   : {top3[0]}, {top3[1]}, {top3[2]}')

print('\n' + '=' * 60)
print('Project complete. All charts saved as PNG files.')
print('=' * 60)

---

## Project Summary

| Item | Detail |
|---|---|
| **Dataset** | `netflix_customer_churn.csv` — 5,000 rows, 14 columns |
| **Target Variable** | `churned` (0 = Active, 1 = Churned) |
| **Models Trained** | Logistic Regression, Decision Tree, Random Forest |
| **Best Model** | Random Forest (highest ROC-AUC) |
| **Key Predictors** | `last_login_days`, `watch_hours`, `avg_watch_time_per_day` |
| **Internship Program** | IBM SkillsBuild Data Analytics with AI |
| **Submitted Under** | BharatCares & AICTE |

---
*End of Project — Netflix Customer Churn Prediction and Data Analysis*